# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [ ]:
import pandas as pd

chunks = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
first_chunk = next(chunks)
print(len(first_chunk))

# Just for testing
first_chunk.head()


In [ ]:
df = first_chunk
df["user_id"] = pd.factorize(df["reviewerID"])[0]
df["item_id"] = pd.factorize(df["asin"])[0]
df["interaction"] = 1


In [ ]:
df = df.sort_values(["user_id", "unixReviewTime"]).copy()

df_test = df.groupby("user_id").tail(1).copy()
df_train = df.drop(df_test.index).copy()

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# MostPopular section

In [ ]:
item_popularity = (
    df_train.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

popular_items = item_popularity.index.tolist()
user_seen_train = df_train.groupby("user_id")["item_id"].apply(set).to_dict()

print(popular_items)

In [ ]:
import math


def recommend_most_popular(user_id, popular_items, user_seen_train, k=10):
    seen = user_seen_train.get(user_id, set())
    recs = []
    for item in popular_items:
        if item not in seen:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def precision_at_k(recommended_items, ground_truth_item,k):
    return 1.0/k if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0


def map(predictions, ground_truth, k=None):
    total_ap = 0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        hits = 0
        sum_precisions = 0

        for i, item in enumerate(ranked, start=1):
            if item in relevant:
                hits += 1
                sum_precisions += hits / i

        total_ap += sum_precisions / len(relevant)
        users += 1

    return total_ap / users if users > 0 else 0


def mrr(predictions, ground_truth, k=None):
    total = 0
    users = 0

    for user in predictions:
        if user not in ground_truth or len(ground_truth[user]) == 0:
            continue

        ranked = predictions[user][:k] if k else predictions[user]
        relevant = ground_truth[user]

        for rank, item in enumerate(ranked, start=1):
            if item in relevant:
                total += 1 / rank
                break

        users += 1

    return total / users if users > 0 else 0

In [ ]:
K = 10
recalls = []
ndcgs = []
precisions = []
ground_truths = {}
all_recs = {}

for _, row in df_test.iterrows():
    user_id = row["user_id"]
    ground_truth_item = row["item_id"]

    recs = recommend_most_popular(user_id, popular_items, user_seen_train, k=K)
    ground_truths[user_id] = [ground_truth_item]
    all_recs[user_id] = recs
    recalls.append(recall_at_k(recs, ground_truth_item))
    precisions.append(precision_at_k(recs,ground_truth_item,K))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

# PROBABLY NEED TO RE-WRITE THOSE FOR THE REAL THING COZ I REFUSE TO BELIEVE EACH USER WILL ACTUALLY ONLY HAVE ONE GROUND TRUTH ITEM
print(f"Users evaluated: {len(recalls)}")
# metrics to use for ranking specifically: Precision@K and Recall@K
print(f"Precision@{K}: {sum(precisions)/len(precisions):.4f}")
print(f"Recall@{K}: {sum(recalls)/len(recalls):.4f}")
# also nice for within-list ordering checks: NDCG@K, MAP AND MRR
print(f"NDCG@{K}: {sum(ndcgs)/len(ndcgs):.4f}")
print("MAP:", map(all_recs,ground_truths,K))
print("MRR:", mrr(all_recs,ground_truths,K))

## BRP

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from implicit.bpr import BayesianPersonalizedRanking

# Load Data
chunks = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
df = next(chunks).copy()

df = df[["reviewerID", "asin", "unixReviewTime"]].copy()
df["interaction"] = 1.0

# Proper factorization
df["user_id"] = pd.factorize(df["reviewerID"])[0]
df["item_id"] = pd.factorize(df["asin"])[0]

# Leave-One-Out Split
df = df.sort_values(["user_id", "unixReviewTime"]).copy()

df_test = df.groupby("user_id").tail(1).copy()
df_train = df.drop(df_test.index).copy()

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# Remove users that lost all train interactions
train_user_counts = df_train["user_id"].value_counts()
valid_users = set(train_user_counts.index)

df_train = df_train[df_train["user_id"].isin(valid_users)].copy()
df_test = df_test[df_test["user_id"].isin(valid_users)].copy()

# Remove test items unseen in train
valid_items = set(df_train["item_id"].unique())
df_test = df_test[df_test["item_id"].isin(valid_items)].copy()

# Remap IDs
user_ids = sorted(df_train["user_id"].unique())
item_ids = sorted(df_train["item_id"].unique())

user_map = {old: new for new, old in enumerate(user_ids)}
item_map = {old: new for new, old in enumerate(item_ids)}

df_train["user_id"] = df_train["user_id"].map(user_map)
df_train["item_id"] = df_train["item_id"].map(item_map)

df_test["user_id"] = df_test["user_id"].map(user_map)
df_test["item_id"] = df_test["item_id"].map(item_map)

df_train = df_train.dropna().copy()
df_test = df_test.dropna().copy()

df_train["user_id"] = df_train["user_id"].astype(int)
df_train["item_id"] = df_train["item_id"].astype(int)
df_test["user_id"] = df_test["user_id"].astype(int)
df_test["item_id"] = df_test["item_id"].astype(int)

num_users = int(df_train["user_id"].max()) + 1
num_items = int(df_train["item_id"].max()) + 1

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print("Users:", num_users, "Items:", num_items)

# Build Sparse Matrix
user_items_csr = csr_matrix(
    (
        df_train["interaction"].astype(np.float32),
        (df_train["user_id"], df_train["item_id"])
    ),
    shape=(num_users, num_items)
)

item_users_csr = user_items_csr.T.tocsr()

model = BayesianPersonalizedRanking(
    factors=64,
    learning_rate=0.05,
    regularization=0.01,
    iterations=100,
    random_state=42
)

model.fit(item_users_csr, show_progress=True)

print("Model user factors:", model.user_factors.shape)
print("Model item factors:", model.item_factors.shape)

# Evaluation
def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0

K = 10
recalls = []
ndcgs = []

max_user_index = model.user_factors.shape[0] - 1
df_test = df_test[df_test["user_id"] <= max_user_index].copy()

for _, row in df_test.iterrows():
    user_id = int(row["user_id"])
    ground_truth_item = int(row["item_id"])

    recs, scores = model.recommend(
        userid=user_id,
        user_items=user_items_csr[user_id],
        N=K,
        filter_already_liked_items=True
    )

    recs = recs.tolist()
    recalls.append(recall_at_k(recs, ground_truth_item))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

print("\n=== BPR-MF Results ===")
print(f"Users evaluated: {len(recalls)}")
print(f"Recall@{K}: {np.mean(recalls):.4f}")
print(f"NDCG@{K}: {np.mean(ndcgs):.4f}")

# Lights GCN